# 🔍 Notebook 03 — Análisis de Diagnóstico

**Sistema de Denuncias Ambientales · Uruguay · D Empathy Project**

> *¿Por qué ocurre? · Patrones latentes · Correlaciones · Anomalías*

Este notebook corresponde a la **Sección 2** del tablero. Después del descriptivo (¿qué?), el diagnóstico busca **estructura oculta** en los datos:

1. **Correlaciones**: ¿qué departamentos tienen perfiles ambientales parecidos? ¿Qué categorías co-ocurren?
2. **Anomalías temporales**: ¿hay meses con desviación estadísticamente significativa?
3. **Triage de riesgo**: combinar volumen + urgencia + recurrencia en un score único para priorizar.

## 📐 Métodos estadísticos

- **Z-score** para anomalías: para cada mes calculo $z = (x - \mu) / \sigma$. Un mes con $|z| > 1.8$ está fuera del rango normal con ~93% de confianza (asumiendo distribución aproximadamente normal del conteo mensual).
- **Score de riesgo compuesto**: $\text{score} = 0.4 \cdot \text{vol\_norm} + 0.35 \cdot \text{pct\_urg} + 0.25 \cdot \text{pct\_perm}$ — pondero los tres factores con un criterio conservador donde el volumen pesa más que la urgencia, y la urgencia más que la permanencia.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path

DATA_DIR = Path("../data")
df = pd.read_parquet(DATA_DIR / "denuncias.parquet")
print(f"Cargado: {df.shape[0]:,} denuncias")

# 1️⃣ Correlaciones y perfiles ambientales

## 1.1 Heatmap motivo × departamento

Cada celda es el conteo de denuncias para esa combinación. El patrón visual revela *especializaciones*: departamentos donde una categoría es desproporcionadamente alta o baja.

In [ ]:
depts_top = df["departamento"].value_counts().head(10).index.tolist()
pivot = (df[df["departamento"].isin(depts_top)]
         .pivot_table(index="departamento", columns="categoria_label",
                      values="id_denuncia", aggfunc="count", fill_value=0))

fig = go.Figure(go.Heatmap(
    z=pivot.values, x=pivot.columns, y=pivot.index,
    colorscale="Teal",
    hovertemplate="<b>%{y}</b> × <b>%{x}</b><br>%{z} denuncias<extra></extra>",
    text=pivot.values, texttemplate="%{text}",
))
fig.update_layout(title="Conteo de denuncias por departamento × categoría (top 10)",
                  height=500, xaxis_tickangle=-30, plot_bgcolor="white")
fig.show()

## 1.2 Normalización: % por departamento

El conteo crudo está dominado por el tamaño del departamento (Canelones tiene mucho de todo). Normalizando por % se ven mejor las **especializaciones reales**.

In [ ]:
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

fig = go.Figure(go.Heatmap(
    z=pivot_pct.values, x=pivot_pct.columns, y=pivot_pct.index,
    colorscale="RdYlGn_r", zmin=0, zmax=50,
    hovertemplate="<b>%{y}</b><br>%{x}<br>%{z:.1f}%<extra></extra>",
    text=pivot_pct.round(1).values, texttemplate="%{text}",
))
fig.update_layout(title="% de cada categoría por departamento (filas suman 100%)",
                  height=500, xaxis_tickangle=-30, plot_bgcolor="white")
fig.show()

# Promedio nacional
media_nacional = df["categoria_label"].value_counts(normalize=True) * 100
print("\n📊 Distribución nacional (% del total):")
print(media_nacional.round(1).to_string())

## 1.3 Radar chart — un departamento vs. la media nacional

El radar muestra el *perfil* del departamento (% en cada categoría) superpuesto al promedio del país. Las puntas que sobresalen son especializaciones.

In [ ]:
def radar_departamento(df, dept):
    cats = sorted(df["categoria_codigo"].unique())
    labels = [df[df["categoria_codigo"]==c]["categoria_label"].iloc[0] for c in cats]

    dept_data = df[df["departamento"] == dept]
    if len(dept_data) == 0:
        print(f"No hay datos para {dept}")
        return
    dept_pct = [(dept_data["categoria_codigo"]==c).mean()*100 for c in cats]
    nat_pct  = [(df["categoria_codigo"]==c).mean()*100 for c in cats]

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=dept_pct + [dept_pct[0]], theta=labels + [labels[0]],
        fill="toself", name=dept, line_color="#1abc9c",
        fillcolor="rgba(26,188,156,0.2)"))
    fig.add_trace(go.Scatterpolar(
        r=nat_pct + [nat_pct[0]], theta=labels + [labels[0]],
        fill="toself", name="Media nacional", line_color="#58a6ff",
        fillcolor="rgba(88,166,255,0.12)", line_dash="dot"))
    fig.update_layout(title=f"Perfil ambiental: {dept} vs. media nacional",
                      height=450,
                      polar={"radialaxis": {"range": [0, max(max(dept_pct), max(nat_pct))*1.1]}})
    return fig

radar_departamento(df, "Maldonado").show()

In [ ]:
radar_departamento(df, "Montevideo").show()

**Lectura**: Maldonado dispara fuerte en Costa (esperable) y modera en Agua, mientras Montevideo está sobre la media nacional en Aire y Residuos. Esto sí es información sustantiva para diseñar políticas focalizadas.

# 2️⃣ Anomalías temporales (Z-score)

Calculo para cada mes el z-score del conteo. Marco como anomalía cuando $|z| > 1.8$, lo que en una distribución normal corresponde aproximadamente al 7% de los casos extremos.

In [ ]:
monthly = df.groupby(df["timestamp"].dt.to_period("M")).size().reset_index(name="total")
monthly["fecha"] = monthly["timestamp"].dt.to_timestamp()
monthly["zscore"] = (monthly["total"] - monthly["total"].mean()) / monthly["total"].std()
monthly["anomalia"] = monthly["zscore"].abs() > 1.8

print(f"Meses analizados: {len(monthly)}")
print(f"Anomalías detectadas: {monthly['anomalia'].sum()} ({monthly['anomalia'].mean()*100:.1f}%)")

fig = go.Figure()
normal = monthly[~monthly["anomalia"]]
anom = monthly[monthly["anomalia"]]
fig.add_trace(go.Scatter(x=normal["fecha"], y=normal["total"], mode="lines+markers",
                         name="Normal", line_color="#58a6ff", marker_size=5))
fig.add_trace(go.Scatter(x=anom["fecha"], y=anom["total"], mode="markers",
                         name="Anomalía",
                         marker={"color":"#f85149","size":12,"symbol":"diamond"}))
fig.update_layout(title="Detección de anomalías mensuales — Z-score > 1.8",
                  xaxis_title="", yaxis_title="Denuncias / mes",
                  height=400, plot_bgcolor="white")
fig.show()

In [ ]:
print("📋 Top 10 meses anómalos:")
anom_top = monthly[monthly["anomalia"]].sort_values("zscore", key=abs, ascending=False).head(10)
for _, r in anom_top.iterrows():
    icono = "📈" if r["zscore"] > 0 else "📉"
    print(f"  {icono} {r['fecha'].strftime('%Y-%m')}: {int(r['total']):4d} denuncias (z={r['zscore']:+.2f})")

**Interpretación**: los picos positivos en 2017-2018 son consistentes con lo que vimos en el descriptivo. Las anomalías negativas (meses muy bajos) en 2010-2011 corresponden a la etapa inicial del sistema de denuncias del Ministerio (subreporte estructural).

# 3️⃣ Triage y priorización

## 3.1 Score de riesgo ambiental

Para cada combinación `(departamento, categoría)` calculo un score 0-100 que combina:
- **40 pts**: volumen relativo (denuncias / máximo)
- **35 pts**: % de denuncias marcadas como urgentes
- **25 pts**: % de denuncias permanentes

**⚠️ Limitación**: en el histórico `urgencia` y `recurrencia` son NaN, así que esos dos términos suman cero. El score actual es prácticamente *sólo volumen*. Cuando llegue suficiente data del formulario, los 3 componentes se activan.

In [ ]:
# Defensivos contra NaN: si la columna es todo NaN, su mean() es NaN → casteo a 0
urg_num = pd.to_numeric(df["urgencia"], errors="coerce").fillna(0)
rec_str = df["recurrencia"].astype(str)
df2 = df.assign(_urg=urg_num, _rec=rec_str)

agg = (df2.groupby(["departamento", "categoria_label"])
       .agg(total=("id_denuncia", "count"),
            pct_urgente=("_urg", "mean"),
            pct_permanente=("_rec", lambda x: (x == "Permanente").mean()))
       .reset_index())

agg["score_riesgo"] = ((agg["total"] / agg["total"].max() * 40) +
                       (agg["pct_urgente"] * 35) +
                       (agg["pct_permanente"] * 25)).round(1)

top15 = agg.nlargest(15, "score_riesgo")[
    ["departamento", "categoria_label", "total", "pct_urgente", "pct_permanente", "score_riesgo"]
]

def semaforo(s):
    return "🔴" if s >= 60 else ("🟡" if s >= 35 else "🟢")

top15["⚠️"] = top15["score_riesgo"].apply(semaforo)
top15

## 3.2 Matriz urgencia × recurrencia (cuadrantes de priorización)

In [ ]:
df_form = df.dropna(subset=["urgencia", "recurrencia"])
if df_form.empty:
    print("⚠️ Aún no hay datos del formulario con urgencia + recurrencia.")
    print("   La matriz de cuadrantes se llenará cuando lleguen denuncias.")
else:
    agg_q = df_form.groupby(["urgencia", "recurrencia"]).size().reset_index(name="total")
    agg_q["urgencia_label"] = agg_q["urgencia"].map({True: "Urgente", False: "No urgente"})
    pivot_q = agg_q.pivot(index="urgencia_label", columns="recurrencia",
                          values="total").fillna(0)
    fig = go.Figure(go.Heatmap(
        z=pivot_q.values, x=pivot_q.columns, y=pivot_q.index,
        colorscale="Reds", text=pivot_q.astype(int).values,
        texttemplate="%{text}", textfont={"size": 18},
    ))
    fig.update_layout(title="Matriz de triage: urgencia × recurrencia",
                      height=300, plot_bgcolor="white")
    fig.show()

---
## ✅ Cierre

- **Correlaciones**: cada departamento tiene un perfil ambiental distintivo. Maldonado se especializa en costa, Montevideo en aire/ruido, Salto y Paysandú en residuos y agua. Esto debería informar políticas focalizadas.
- **Anomalías**: los picos 2017-2018 dominan la serie. Vale investigar si fue un cambio de criterio del organismo o una campaña pública.
- **Triage**: el score combinado funciona pero está sub-utilizado hasta que el formulario aporte datos de urgencia/recurrencia.

**Próximo notebook**: `04_predictivo.ipynb` — pronósticos, NLP, alertas y clustering.